# 04 — Feature Engineering · Databricks
**Exécuter après** : `03_unification_db.ipynb`
---

In [ ]:
%pip install vaderSentiment scipy -q

In [ ]:
# ============================================================
# Setup Databricks — import spark_utils par chemin absolu
# Évite tout conflit avec le package pip "spark_utils" system
# ============================================================
import sys, os, importlib.util

# 1. Résolution du repo root via Databricks Repos context
_ctx   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_parts = _ctx.split('/')
_idx   = next((i for i, p in enumerate(_parts) if p == 'fakenews-analyzer'), None)

if _idx is None:
    raise RuntimeError(f"'fakenews-analyzer' introuvable dans le path Databricks : {_ctx}")

_REPO       = '/Workspace/' + '/'.join(_parts[1:_idx+1])
_UTILS_FILE = _REPO + '/02_preprocessing/spark_utils.py'

if not os.path.exists(_UTILS_FILE):
    raise FileNotFoundError(
        f"spark_utils.py introuvable : {_UTILS_FILE}
"
        "Vérifier que le repo est bien lié dans Databricks Repos et que le Pull est fait."
    )

# 2. Import direct par fichier (bypass total du sys.path et des packages pip)
_spec = importlib.util.spec_from_file_location('_spark_utils', _UTILS_FILE)
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

load_raw_sources        = _mod.load_raw_sources
show_label_distribution = _mod.show_label_distribution
save_parquet            = _mod.save_parquet
load_parquet            = _mod.load_parquet
stratified_split        = _mod.stratified_split
check_class_balance     = _mod.check_class_balance

# 3. Chemins Volume Unity Catalog
# → Databricks > Data > Volumes > clic droit "fakenews" > Copy path
VOLUME        = '/Volumes/main/default/fakenews'  # <- MODIFIER si catalog/schema différent

RAW_DIR       = VOLUME + '/raw'
PROCESSED_DIR = VOLUME + '/processed'
MODELS_DIR    = VOLUME + '/models'
COLAB_DIR     = VOLUME + '/colab_exports'
REPORTS_DIR   = VOLUME + '/reports'

for d in [PROCESSED_DIR, MODELS_DIR, COLAB_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Repo       : {_REPO}')
print(f'spark_utils: {_UTILS_FILE}  ✓')
print(f'Volume     : {VOLUME}')
print('Setup OK')

In [ ]:
import numpy as np
import scipy.sparse as sp
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType, StructType, StructField
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF
print('Imports OK')

## Section 1 — Chargement des données

In [ ]:
train_df = load_parquet(spark, os.path.join(PROCESSED_DIR, 'train.parquet'))
test_df  = load_parquet(spark, os.path.join(PROCESSED_DIR, 'test.parquet'))

if train_df is None or test_df is None:
    raise FileNotFoundError('train.parquet ou test.parquet introuvable. Executer 03_unification_db.ipynb.')

print(f'Train : {train_df.count():,} lignes')
print(f'Test  : {test_df.count():,} lignes')
display(train_df.limit(3))

## Section 2 — Pipeline TF-IDF Spark

In [ ]:
tokenizer  = RegexTokenizer(inputCol='clean_text', outputCol='tokens_raw', pattern=r'\W+', minTokenLength=3)
remover    = StopWordsRemover(inputCol='tokens_raw', outputCol='tokens',
                               stopWords=StopWordsRemover.loadDefaultStopWords('english'))
hashing_tf = HashingTF(inputCol='tokens', outputCol='tf_features', numFeatures=50_000)
idf        = IDF(inputCol='tf_features', outputCol='tfidf_features', minDocFreq=2)

tfidf_pipeline = Pipeline(stages=[tokenizer, remover, hashing_tf, idf])

print('Entrainement du pipeline TF-IDF sur le train...')
tfidf_model    = tfidf_pipeline.fit(train_df)
train_features = tfidf_model.transform(train_df)
test_features  = tfidf_model.transform(test_df)
print('Pipeline TF-IDF entraine')

In [ ]:
tfidf_path = os.path.join(MODELS_DIR, 'baseline', 'tfidf_vectorizer')
os.makedirs(tfidf_path, exist_ok=True)
tfidf_model.write().overwrite().save(tfidf_path)
print(f'Modele TF-IDF sauvegarde : {tfidf_path}')

## Section 3 — Features NLP (VADER Sentiment)

In [ ]:
def extract_nlp_features(text: str):
    if not text:
        return (0, 0.0, 0.0, 0, 0)
    word_count    = len(text.split())
    alpha_chars   = [c for c in text if c.isalpha()]
    capital_ratio = sum(1 for c in alpha_chars if c.isupper()) / len(alpha_chars) if alpha_chars else 0.0
    exclamation   = text.count('!')
    question      = text.count('?')
    sentiment_score = 0.0
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        analyzer        = SentimentIntensityAnalyzer()
        sentiment_score = analyzer.polarity_scores(text)['compound']
    except Exception:
        pass
    return (word_count, float(sentiment_score), float(capital_ratio), exclamation, question)

nlp_schema = StructType([
    StructField('word_count',        IntegerType()),
    StructField('sentiment_score',   FloatType()),
    StructField('capital_ratio',     FloatType()),
    StructField('exclamation_count', IntegerType()),
    StructField('question_count',    IntegerType())
])

nlp_udf = F.udf(extract_nlp_features, nlp_schema)

train_features = train_features.withColumn('nlp', nlp_udf(F.col('clean_text')))
test_features  = test_features.withColumn('nlp',  nlp_udf(F.col('clean_text')))

for field in ['word_count', 'sentiment_score', 'capital_ratio', 'exclamation_count', 'question_count']:
    train_features = train_features.withColumn(field, F.col(f'nlp.{field}'))
    test_features  = test_features.withColumn(field,  F.col(f'nlp.{field}'))

train_features = train_features.drop('nlp')
test_features  = test_features.drop('nlp')
print('Features NLP calculees')

## Section 4 — Export pour Google Colab (DistilBERT)

In [ ]:
def spark_vector_to_scipy(df, feature_col, label_col, n_features=50_000):
    from pyspark.ml.linalg import SparseVector
    rows   = df.select(feature_col, label_col).collect()
    labels = np.array([row[label_col] for row in rows])
    data_rows = []
    for row in rows:
        vec = row[feature_col]
        if hasattr(vec, 'toArray'):
            data_rows.append(vec.toArray())
        else:
            arr = np.zeros(n_features)
            if hasattr(vec, 'indices') and hasattr(vec, 'values'):
                arr[vec.indices] = vec.values
            data_rows.append(arr)
    return sp.csr_matrix(np.vstack(data_rows)), labels

print('Export des features TF-IDF pour Colab (peut prendre quelques minutes)...')
X_train, y_train = spark_vector_to_scipy(train_features, 'tfidf_features', 'label')
X_test,  y_test  = spark_vector_to_scipy(test_features,  'tfidf_features', 'label')

sp.save_npz(os.path.join(COLAB_DIR, 'train_features.npz'), X_train)
sp.save_npz(os.path.join(COLAB_DIR, 'test_features.npz'),  X_test)
np.save(os.path.join(COLAB_DIR, 'train_labels.npy'), y_train)
np.save(os.path.join(COLAB_DIR, 'test_labels.npy'),  y_test)
print(f'Features exportees : X_train={X_train.shape} | X_test={X_test.shape}')

In [ ]:
# Exporter les textes bruts pour DistilBERT (Google Colab)
train_texts = train_df.select('clean_text', 'label').toPandas()
test_texts  = test_df.select('clean_text',  'label').toPandas()

train_texts.to_csv(os.path.join(COLAB_DIR, 'train_texts.csv'), index=False)
test_texts.to_csv(os.path.join(COLAB_DIR,  'test_texts.csv'),  index=False)

print(f'Textes exportes : {len(train_texts):,} train | {len(test_texts):,} test')
print(f'\nFichiers dans {COLAB_DIR} :')
for item in dbutils.fs.ls(ff'dbfs:{VOLUME}/colab_exports/'):
    print(f'  {item.name:<35} {item.size / 1024 / 1024:.1f} MB')

print('\nProchaine etape : 05_baseline_tfidf_db.ipynb')